In [ ]:
import crewai.llms.cache as crewai_cache

print("Before patch:", crewai_cache.mark_cache_breakpoint)

crewai_cache.mark_cache_breakpoint = lambda msg: msg

print("After patch:", crewai_cache.mark_cache_breakpoint)

In [ ]:
!pip install litellm

In [ ]:
!pip install crewai

In [ ]:
import warnings
warnings.filterwarnings('ignore')


In [ ]:
from crewai import Task,Agent,Crew
import os

In [ ]:

print(bool(os.getenv("GROQ_API_KEY")))


In [ ]:
import sys
import litellm


In [ ]:
import crewai.llms.cache as _crewai_cache

_crewai_cache.mark_cache_breakpoint = lambda msg: msg

In [ ]:
from crewai import LLM

llm = LLM(
    model="groq/openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens=1000
)

In [ ]:
%pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

vector = embedding_model.encode("Hello, this is a memory.")
print(len(vector))

In [ ]:
from crewai.memory import Memory

def my_embedding_function(texts):
    return embedding_model.encode(texts).tolist()

memory = Memory(
    llm=llm,
    embedder=my_embedding_function
)
print(memory)

In [ ]:
response = llm.call("Say hello in one sentence.")
print(response)

In [ ]:
customer_support_agent = Agent(
    role="Senior Customer Support Representative",
    goal="Be the most friendly and helpful "
         "customer support representative in your team",
    backstory=(
    "You are a customer support representative at CrewAI. "
    "Give accurate, clear, and concise answers."
    ),
    allow_delegation=False,
    llm=llm,
    verbose=True
)


In [ ]:
quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the "
         "best support quality assurance in your team",
    backstory=(
    "You are a customer support representative at CrewAI. "
    "Give accurate, clear, and concise answers."
     ),
    llm=llm,
    verbose=True
)


In [ ]:
!pip install crewai-tools

In [ ]:
import crewai
print(crewai.__version__)

In [ ]:
from crewai_tools import SerperDevTool,ScrapeWebsiteTool,WebsiteSearchTool


In [ ]:
#SerpDevToll->do google search and get relevant searhch results
#scrapeWebsiteTool->web scraping and data collection
#WebsiteSearchToll->semantic searches within website content


In [ ]:
docs_scrape_tool=ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)


In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} reached out with this inquiry:\n"
        "{inquiry}\n\n"
        "{person} from {customer} needs help. "
        "Give an accurate and useful answer."
    ),
    expected_output=(
        "A clear answer under 250 words. "
        "Include only the important steps and code needed."
    ),
    agent=customer_support_agent,
)

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response from the senior support representative "
        "for {customer}'s inquiry.\n"
        "Check whether it is accurate, complete, and easy to understand. "
        "Correct any important mistakes or missing information."
    ),

    expected_output=(
        "A corrected answer under 200 words."
    ),

    agent=quality_assurance_agent,
)

In [ ]:
#creating crew
crew = Crew(
    agents=[customer_support_agent,quality_assurance_agent],
    tasks=[inquiry_resolution,quality_assurance_review],
    tools=[docs_scrape_tool],
    verbose=True,
    memory=memory
)


In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}

from IPython.display import Markdown, display

result = await crew.kickoff_async(inputs=inputs)

display(Markdown(result.raw))
